### Groker – Lösung Schritt für Schritt

*34. Bundeswettbewerb Informatik (2015), 1. Runde, Aufgabe 5*

Groker wird zu zweit gespielt. Jeder Spieler hat einen Vorrat von genügend vielen Chips und einen Gewinnhaufen, der zu Beginn leer ist. Das Spiel wird in mehreren Runden gespielt. In jeder Runde setzen die Spieler jeweils für den Gegenspieler verdeckt eine beliebige Anzahl Chips – ihren Einsatz –, mindestens aber einen Chip. Aufgedeckt wird dann gleichzeitig.

Unterscheidet sich die Zahl der gesetzten Chips um höchstens fünf, dürfen beide Spieler ihren Einsatz komplett dem eigenen Gewinnhaufen hinzufügen. Ist die Differenz größer, darf lediglich derjenige mit dem kleineren Einsatz die Chips seinem Gewinnhaufen hinzufügen.

Derjenige Spieler hat gewonnen, der in seinem Gewinnhaufen als erster 100 Chips mehr als sein Gegenspieler hat.

Ein Spiel könnte also z. B. so ablaufen; Spieler 2 gewinnt:

| Einsatz Spieler 1 | Einsatz Spieler 2 | Gewinnhaufen Spieler 1 | Gewinnhaufen Spieler 2 |
|---|---|---|---|
| 66 | 61 | 66 | 61 |
| 66 | 60 | 66 | 121 |
| 53 | 47 | 66 | 168 |

Zur Entwicklung von Spielstrategien kann man sich überlegen, dass Einsätze zwischen einem und sechs Chips sicher sind, da der Gegenspieler ja mindestens einen Chip setzen muss. Immer sechs Chips zu setzen wäre also eine mögliche Strategie. Wenn der Gegner das merkt, kann dieser aber permanent elf Chips setzen, seinen Vorsprung pro Runde also um fünf Chips vergrößern. Insofern könnte es sinnvoll sein, zufällig weniger Chips zu setzen, damit der Gegner eventuell in dieser Runde gar nichts bekommt.

#### Aufgabe 5

Schreibe ein Programm, das im BwInf-Turniersystem (turnier.bundeswettbewerb-informatik.de) gegen andere Programme Groker spielt und möglichst häufig gewinnt.

**Zu den Daten:** Beispieldaten gibt es nicht. Das Turniersystem gibt es nicht mehr; als Gegner programmieren wir deshalb selbst einige Strategien und lassen sie gegeneinander spielen.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Simulation:** Wir bauen das Spiel nach (Regeln als Funktion, Spielverlauf als Schleife) und lassen Strategien gegeneinander antreten.
- **Auszahlungsmatrix:** Die Einzelrunde ist ein Spiel mit gleichzeitiger Wahl. Für jedes Paar von Einsätzen weiß man, wie viele Chips der eine mehr bekommt als der andere.
- **Gemischte Strategie:** Setzt man immer dasselbe, kann der Gegner sich darauf einstellen. Wählt man den Einsatz zufällig mit passenden Wahrscheinlichkeiten, hat der Gegner keine feste Gegenstrategie, die im Schnitt gewinnt (Gleichgewicht).
- **Beste Antwort:** Kennt man ungefähr die Wahrscheinlichkeiten für den gegnerischen Einsatz, rechnet man für jeden eigenen Einsatz den erwarteten Gewinn aus und nimmt den größten (adaptive Strategie).
- **Gewichteter Mittelwert:** Neuere Einsätze zählen mehr als alte, damit man auf wechselndes Verhalten des Gegners reagiert.
- **Fictitious play:** Ein einfaches Näherungsverfahren für das Gleichgewicht: Man setzt immer die beste Antwort auf die bisherige Häufigkeit der eigenen Wahl (das Spiel ist symmetrisch).

**Nützliche Funktionen und Bibliotheken**
- `random.seed`, `random.randint(a, b)`: Zufallszahlen (mit `seed` jedes Mal dieselben).
- `random.choices(liste, weights=gewichte)`: wählt ein Element mit vorgegebenen Wahrscheinlichkeiten.
- `random.choice(liste)`: wählt ein zufälliges Element.

## Die Idee in Worten

Die Aufgabe ist offen: Es gibt keine „richtige“ Ausgabe, sondern eine KI, die im Turnier gut abschneidet. Das Turniersystem existiert nicht mehr, also bauen wir uns einen kleinen Spielrahmen und eigene Gegner.

1. **Regeln und Spielablauf** als Funktionen `runde` und `spiel`. Eine Strategie bekommt die bisherigen Einsätze beider Spieler und liefert den nächsten Einsatz.
2. **Strategien aus den Hinweisen**: konstant 6, immer 5 mehr, immer 6 weniger, Durchschnitt, Zufall. Wir zeigen ihre Schwächen.
3. **Gleichgewichtsstrategie**: Wir rechnen die Auszahlungen aus und suchen Wahrscheinlichkeiten für die Einsätze, gegen die kein fester Einsatz im Schnitt gewinnt.
4. **Adaptive Strategie**: Sie schätzt aus den letzten Einsätzen des Gegners, was er als Nächstes setzt, und spielt die beste Antwort darauf.
5. **Turnier** jeder gegen jeden.

Da nur ein Vorsprung von 100 Chips zählt, kommt es auf den Gewinn **pro Runde** an: Wer im Schnitt pro Runde mehr bekommt als der Gegner, gewinnt irgendwann.

## Schritt 1: Die Regeln

`runde(a, b)` gibt zurück, wie viele Chips die beiden Spieler bei den Einsätzen `a` und `b` zu ihrem Haufen bekommen. Die Auszahlung `auszahlung(a, b)` ist der Unterschied aus Sicht des ersten Spielers, denn nur der Unterschied der Haufen entscheidet das Spiel.

Wir prüfen die Tabelle aus der Aufgabe: Bei 66 gegen 61 (Differenz 5) bekommen beide ihren Einsatz, bei 66 gegen 60 und 53 gegen 47 (Differenz 6) nur der Kleinere.

In [ ]:
def runde(a, b):
    """Zuwachs der Gewinnhaufen (Spieler 1, Spieler 2) bei den Einsätzen a und b."""
    if abs(a - b) <= 5:
        return a, b                 # beide dürfen ihren Einsatz behalten
    if a < b:
        return a, 0                 # nur der kleinere Einsatz zählt
    return 0, b


def auszahlung(a, b):
    """Wie viele Chips Spieler 1 mehr bekommt als Spieler 2."""
    x, y = runde(a, b)
    return x - y


haufen1 = haufen2 = 0
for a, b in [(66, 61), (66, 60), (53, 47)]:
    x, y = runde(a, b)
    haufen1 += x
    haufen2 += y
    print(a, b, '->', haufen1, haufen2)

66 61 -> 66 61
66 60 -> 66 121
53 47 -> 66 168


Das stimmt mit der Tabelle der Aufgabe überein (66/61, 66/121, 66/168). Jetzt prüfen wir die Aussagen zu den „sicheren“ Einsätzen: Mit 1 bis 6 Chips bekommt man seinen Einsatz sicher, denn der Gegner setzt mindestens 1. Wir rechnen für einige Einsätze aus, wie viele Chips man sicher bekommt und wie weit man schlimmstenfalls hinter dem Gegner liegt.

In [ ]:
for a in [1, 6, 7, 11, 12, 20]:
    sicher = min(runde(a, b)[0] for b in range(1, 200))
    schlechtester = min(auszahlung(a, b) for b in range(1, 200))
    print('Einsatz', a, ': sicher', sicher, 'Chips, schlimmstenfalls', schlechtester, 'gegenüber dem Gegner')

Einsatz 1 : sicher 1 Chips, schlimmstenfalls -5 gegenüber dem Gegner
Einsatz 6 : sicher 6 Chips, schlimmstenfalls -5 gegenüber dem Gegner
Einsatz 7 : sicher 0 Chips, schlimmstenfalls -5 gegenüber dem Gegner
Einsatz 11 : sicher 0 Chips, schlimmstenfalls -5 gegenüber dem Gegner
Einsatz 12 : sicher 0 Chips, schlimmstenfalls -6 gegenüber dem Gegner
Einsatz 20 : sicher 0 Chips, schlimmstenfalls -14 gegenüber dem Gegner


Bis 6 Chips bekommt man den Einsatz sicher, der Gegner kann aber bis zu 5 Chips mehr bekommen (mit 11). Ab 7 kann man leer ausgehen (der Gegner setzt 1 und bekommt es allein). Ab 12 kann der Gegner sogar 6 oder mehr Chips Vorsprung holen (er setzt 6 weniger und bekommt als Einziger etwas). Mehr als 11 zu setzen lohnt sich daher nicht, die Hinweise sagen dasselbe. Für den Rest der Aufgabe reichen Einsätze von 1 bis 14.

## Schritt 2: Der Spielablauf

Eine Strategie ist eine Funktion `setze(name, meine, gegner)`: Sie bekommt den Namen der Strategie, die bisherigen eigenen Einsätze und die bisherigen Einsätze des Gegners (als Listen) und liefert den nächsten Einsatz. Die Strategien kommen unten einzeln dazu; `setze` ruft die passende auf. `spiel` lässt zwei Strategien gegeneinander spielen, bis einer 100 Chips mehr hat. Damit kein Spiel endlos läuft, brechen wir nach 2000 Runden ohne Sieger ab.

*Hinweis:* Wir übergeben keine Funktionen, sondern nur den Namen der Strategie als Text; `setze` unterscheidet mit `if`.

In [ ]:
import random


def spiel(name1, name2, max_runden=2000):
    """Spielt bis zu einem Vorsprung von 100. Rückgabe: Sieger (1, 2 oder 0 bei Abbruch), Rundenzahl, Einsatzlisten."""
    haufen1 = haufen2 = 0
    meine1 = []
    meine2 = []
    for r in range(1, max_runden + 1):
        a = setze(name1, meine1, meine2)
        b = setze(name2, meine2, meine1)
        x, y = runde(a, b)
        haufen1 += x
        haufen2 += y
        meine1.append(a)
        meine2.append(b)
        if haufen1 - haufen2 >= 100:
            return 1, r, meine1, meine2
        if haufen2 - haufen1 >= 100:
            return 2, r, meine1, meine2
    return 0, max_runden, meine1, meine2


def zeige_spiel(name1, name2, anzahl=8):
    """Gibt die ersten Runden eines Spiels als Tabelle aus."""
    sieger, runden, m1, m2 = spiel(name1, name2)
    h1 = h2 = 0
    print('Einsatz 1  Einsatz 2  Haufen 1  Haufen 2')
    for r in range(min(anzahl, runden)):
        x, y = runde(m1[r], m2[r])
        h1 += x
        h2 += y
        print('%9d  %9d  %8d  %8d' % (m1[r], m2[r], h1, h2))
    print('Sieger: Spieler', sieger, 'nach', runden, 'Runden')

## Schritt 3: Einfache Strategien aus den Hinweisen

Die Hinweise nennen diese Strategien, sie dienen uns als Gegner:

- `konstant6`: immer 6 (sicher, aber leicht zu schlagen).
- `konstant11`: immer 11 (zum Vorführen, schlägt `konstant6`).
- `fuenfmehr`: immer 5 mehr als der letzte Einsatz des Gegners (schlägt feste Einsätze).
- `sechsweniger`: immer 6 weniger als der letzte Einsatz des Gegners, bei Einsätzen unter 7 stattdessen 5 mehr.
- `durchschnitt`: wie `sechsweniger`, aber bezogen auf den gerundeten Durchschnitt aller bisherigen Einsätze des Gegners.
- `zufall`: gleichverteilt 1 bis 11 (die Hinweise: Zufall zwischen `a ≤ 6` und `a + 10`).

Die erste Runde setzen alle auf 6.

In [ ]:
def setze_einfach(name, meine, gegner):
    if not gegner:
        return 6                                    # erste Runde
    if name == 'konstant6':
        return 6
    if name == 'konstant11':
        return 11
    if name == 'fuenfmehr':
        return gegner[-1] + 5
    if name == 'sechsweniger':
        letzter = gegner[-1]
        if letzter >= 7:
            return letzter - 6
        return letzter + 5
    if name == 'durchschnitt':
        mittel = round(sum(gegner) / len(gegner))
        if mittel >= 7:
            return mittel - 6
        return mittel + 5
    if name == 'zufall':
        return random.randint(1, 11)


def setze(name, meine, gegner):
    return setze_einfach(name, meine, gegner)


random.seed(1)
zeige_spiel('konstant6', 'konstant11', 5)

Einsatz 1  Einsatz 2  Haufen 1  Haufen 2
        6          6         6         6
        6         11        12        17
        6         11        18        28
        6         11        24        39
        6         11        30        50
Sieger: Spieler 2 nach 21 Runden


Gegen immer 11 holt man mit 6 keinen Vorsprung: Der Gegner bekommt pro Runde 5 Chips mehr, nach 20 Runden mit Differenz 5 ist sein Vorsprung 100 (die erste Runde ist ausgeglichen, das Spiel dauert also 21 Runden). Die beiden folgenden Spiele zeigen die Gegenstrategien aus den Hinweisen: `fuenfmehr` gegen `konstant6` und `sechsweniger` gegen `fuenfmehr`.

In [ ]:
zeige_spiel('fuenfmehr', 'konstant6', 5)
print()
zeige_spiel('sechsweniger', 'fuenfmehr', 5)

Einsatz 1  Einsatz 2  Haufen 1  Haufen 2
        6          6         6         6
       11          6        17        12
       11          6        28        18
       11          6        39        24
       11          6        50        30
Sieger: Spieler 1 nach 21 Runden

Einsatz 1  Einsatz 2  Haufen 1  Haufen 2
        6          6         6         6
       11         11        17        17
        5         16        22        17
       10         10        32        27
        4         15        36        27
Sieger: Spieler 1 nach 73 Runden


`fuenfmehr` setzt gegen `konstant6` immer 11 und gewinnt jede Runde mit 5 Chips Vorsprung. Gegen `sechsweniger` verliert `fuenfmehr` dagegen: Nach dem ersten Wechsel setzt `fuenfmehr` 16, `sechsweniger` aber nur 5 (6 weniger als 11). Der Abstand ist dann 11, nur `sechsweniger` bekommt etwas. In der folgenden Runde setzen beide 10, danach wieder 15 gegen 4, und so weiter: Immer wenn die Einsätze weit auseinander liegen, gewinnt der kleinere. Jede feste Regel hat eine Gegenregel.

## Schritt 4: Die Gleichgewichtsstrategie

Gegen jede Regel, die nur vom letzten Einsatz abhängt, gibt es eine Gegenregel. Gegen **zufällige** Einsätze gibt es keine feste Gegenregel. Wir suchen deshalb Wahrscheinlichkeiten `p[1], …, p[14]` für die Einsätze, sodass **jeder feste Einsatz** gegen diese Mischung im Schnitt höchstens 0 Chips pro Runde mehr bekommt als die Mischung.

Das Spiel ist symmetrisch, also ist der Wert 0 (beide haben die gleichen Möglichkeiten). Wir finden die Mischung mit *fictitious play*: Wir führen eine Zählliste der bisher gewählten Einsätze. In jedem Schritt berechnen wir den Einsatz, der gegen die Häufigkeiten der Zählliste die höchste Auszahlung hat, und zählen ihn dazu. Die relativen Häufigkeiten laufen bei solchen Spielen gegen ein Gleichgewicht.

In [ ]:
MAX_EINSATZ = 14

# Auszahlungstabelle: tabelle[e - 1][b - 1] ist die Auszahlung von Einsatz e gegen Einsatz b
tabelle = [[auszahlung(e, b) for b in range(1, MAX_EINSATZ + 1)] for e in range(1, MAX_EINSATZ + 1)]

zaehler = [1] * MAX_EINSATZ                      # Einsatz e wird in zaehler[e - 1] gezählt
for schritt in range(10000):
    beste_e = 1
    bester_wert = None
    for e in range(1, MAX_EINSATZ + 1):
        wert = 0
        for b in range(1, MAX_EINSATZ + 1):
            wert += zaehler[b - 1] * tabelle[e - 1][b - 1]
        if bester_wert is None or wert > bester_wert:
            bester_wert = wert
            beste_e = e
    zaehler[beste_e - 1] += 1

summe = sum(zaehler)
gleichgewicht = [z / summe for z in zaehler]
for e in range(1, MAX_EINSATZ + 1):
    print('Einsatz', e, ':', round(gleichgewicht[e - 1], 3))

Einsatz 1 : 0.0
Einsatz 2 : 0.113
Einsatz 3 : 0.1
Einsatz 4 : 0.075
Einsatz 5 : 0.068
Einsatz 6 : 0.063
Einsatz 7 : 0.06
Einsatz 8 : 0.129
Einsatz 9 : 0.111
Einsatz 10 : 0.099
Einsatz 11 : 0.099
Einsatz 12 : 0.083
Einsatz 13 : 0.0
Einsatz 14 : 0.0


Wir prüfen die Mischung: Für jeden festen Einsatz `e` rechnen wir die durchschnittliche Auszahlung gegen die Mischung aus. Bei einem echten Gleichgewicht wäre das nie größer als 0.

In [ ]:
werte = []
for e in range(1, MAX_EINSATZ + 1):
    wert = 0
    for b in range(1, MAX_EINSATZ + 1):
        wert += gleichgewicht[b - 1] * tabelle[e - 1][b - 1]
    werte.append(wert)
print('Auszahlung der festen Einsätze 1 bis 14 gegen die Mischung:')
print([round(w, 3) for w in werte])
print('größter Wert:', round(max(werte), 3))

Auszahlung der festen Einsätze 1 bis 14 gegen die Mischung:
[-0.544, 0.035, 0.005, 0.008, 0.021, -0.066, -0.063, 0.028, 0.01, 0.046, 0.009, -0.101, -0.302, -1.583]
größter Wert: 0.046


Der größte Wert liegt nahe bei 0: Kein fester Einsatz gewinnt gegen die Mischung im Schnitt mehr als ein paar Hundertstel Chips pro Runde (mit mehr Schritten würde der Wert noch kleiner). Die Mischung benutzt nur Einsätze von 2 bis 12. Am häufigsten sind 8 und 2 bis 3; die 1 und alles ab 13 kommen nicht vor. Die zwei „Gipfel“ sorgen dafür, dass man weder durch „6 weniger“ noch durch „5 mehr“ systematisch geschlagen wird.

Die Strategie `gleichgewicht` zieht einen Einsatz mit diesen Wahrscheinlichkeiten. `random.choices(liste, weights=...)` macht das.

In [ ]:
def setze_gleichgewicht():
    return random.choices(range(1, MAX_EINSATZ + 1), weights=gleichgewicht)[0]


def setze(name, meine, gegner):
    if name == 'gleichgewicht':
        return setze_gleichgewicht()
    return setze_einfach(name, meine, gegner)


random.seed(2)
zeige_spiel('gleichgewicht', 'konstant6', 5)

Einsatz 1  Einsatz 2  Haufen 1  Haufen 2
       12          6         0         6
       12          6         0        12
        2          6         2        18
        2          6         4        24
       11          6        15        30
Sieger: Spieler 1 nach 1382 Runden


## Schritt 5: Die adaptive Strategie

Ein Gleichgewicht ist ein sicherer Schutz, aber es nutzt Schwächen des Gegners nicht aus: Gegen `konstant6` bekommt es nur im Schnitt etwas mehr. Die **adaptive Strategie** versucht, das Verhalten des Gegners zu lernen:

1. Aus den letzten 20 Einsätzen des Gegners wird eine **gewichtete Häufigkeit** gebildet: der letzte Einsatz zählt 1, der davor 0,85, der davor 0,85², usw. Neuere Einsätze zählen also mehr, damit wir auf Wechsel der Gegnerstrategie reagieren.
2. Für jeden eigenen Einsatz `e` von 1 bis 14 wird die **erwartete Auszahlung** gegen diese Häufigkeiten ausgerechnet.
3. Wir setzen den Einsatz mit der größten erwarteten Auszahlung. Bei mehreren gleich guten wählen wir zufällig, damit wir nicht berechenbar werden.
4. Solange wir weniger als 5 Einsätze des Gegners kennen, spielen wir das Gleichgewicht.

Damit erkennt die Strategie feste Einsätze (`konstant6`) und Zufallsverteilungen (`zufall`). Wer abwechselnd viel und wenig setzt oder sich nach unserem Einsatz richtet, kann sie aber durcheinanderbringen (das Turnier unten zeigt, dass sie gegen das Gleichgewicht nicht gewinnt).

In [ ]:
def beste_antwort(gegner):
    """Einsatz mit der größten erwarteten Auszahlung gegen die gewichtete Häufigkeit der letzten gegnerischen Einsätze."""
    gewicht = {}
    g = 1.0
    for b in reversed(gegner[-20:]):
        gewicht[b] = gewicht.get(b, 0) + g
        g = g * 0.85
    beste = []
    bester_wert = None
    for e in range(1, MAX_EINSATZ + 1):
        wert = 0
        for b in gewicht:
            wert += gewicht[b] * auszahlung(e, b)
        if bester_wert is None or wert > bester_wert + 1e-9:
            bester_wert = wert
            beste = [e]
        elif abs(wert - bester_wert) <= 1e-9:
            beste.append(e)
    return random.choice(beste)


def setze(name, meine, gegner):
    if name == 'gleichgewicht':
        return setze_gleichgewicht()
    if name == 'adaptiv':
        if len(gegner) < 5:
            return setze_gleichgewicht()
        return beste_antwort(gegner)
    return setze_einfach(name, meine, gegner)


random.seed(3)
zeige_spiel('adaptiv', 'konstant6', 8)

Einsatz 1  Einsatz 2  Haufen 1  Haufen 2
        4          6         4         6
        8          6        12        12
        6          6        18        18
        8          6        26        24
        9          6        35        30
       11          6        46        36
       11          6        57        42
       11          6        68        48
Sieger: Spieler 1 nach 24 Runden


Gegen `konstant6` spielt `adaptiv` die ersten Runden wie das Gleichgewicht, erkennt dann, dass der Gegner immer 6 setzt, und antwortet ab der sechsten Runde mit 11: Das sind 5 Chips mehr pro Runde als der Gegner.

## Schritt 6: Das Turnier

Jede Strategie spielt gegen jede andere 10 Spiele. Die Tabelle zeigt, wie viele davon die Strategie in der **Zeile** gegen die Strategie in der **Spalte** gewonnen hat (Spiele ohne Sieger nach 2000 Runden zählen für keinen). Am Ende steht die Summe der Siege (maximal 60).

In [ ]:
namen = ['konstant6', 'fuenfmehr', 'sechsweniger', 'zufall', 'durchschnitt', 'gleichgewicht', 'adaptiv']
SPIELE = 10

random.seed(42)
siege = {}                                      # (Zeile, Spalte) -> Siege der Zeile
unentschieden = 0
for n1 in namen:
    for n2 in namen:
        if n1 == n2:
            continue
        s = 0
        for i in range(SPIELE):
            sieger = spiel(n1, n2)[0]
            if sieger == 1:
                s += 1
            if sieger == 0:
                unentschieden += 1
        siege[(n1, n2)] = s

print('%-14s' % '', ' '.join('%5s' % n[:5] for n in namen), ' Summe')
for n1 in namen:
    zeile = []
    for n2 in namen:
        if n1 == n2:
            zeile.append('    -')
        else:
            zeile.append('%5d' % siege[(n1, n2)])
    gesamt = sum(siege[(n1, n2)] for n2 in namen if n2 != n1)
    print('%-14s' % n1, ' '.join(zeile), '%5d' % gesamt)
print('Spiele ohne Sieger:', unentschieden)

               konst fuenf sechs zufal durch gleic adapt  Summe
konstant6          -     0     0     1     0     3     0     4
fuenfmehr         10     -     0     0    10     0     0    20
sechsweniger      10    10     -     5    10     2     0    37
zufall             6    10     6     -     7     4     1    34
durchschnitt      10     0     0     3     -     1     0    14
gleichgewicht      8    10     4     6    10     -     6    44
adaptiv           10    10    10    10    10     3     -    53
Spiele ohne Sieger: 18


**Was das Turnier zeigt** (die Zahlen gelten für den Startwert 42 und schwanken mit dem Zufall):

- `adaptiv` hat mit 53 von 60 möglichen Siegen die meisten. Gegen alle einfachen Strategien gewinnt er alle 10 Spiele, nur gegen das Gleichgewicht bleibt er bei 3 Siegen (das Gleichgewicht gewinnt 6 von 10).
- `gleichgewicht` ist die zweitbeste Strategie (44). Gegen die Einfachen gewinnt es oft, aber nicht immer: Es nutzt feste Schwächen kaum aus, sein Vorteil pro Runde ist klein. Gegen `konstant6` dauert ein Spiel manchmal über tausend Runden, und 18 Spiele endeten ohne Sieger.
- `sechsweniger` (37) und `zufall` (34) liegen im Mittelfeld. `sechsweniger` schlägt `fuenfmehr`, `konstant6` und `durchschnitt` klar, verliert aber gegen `adaptiv` jedes Mal.
- `fuenfmehr` (20), `durchschnitt` (14) und `konstant6` (4) sind schwach, wie in den Hinweisen angekündigt: Jede wird von einer Gegenregel geschlagen.

Als KI für das Turnier würden wir also `adaptiv` nehmen: Er gewinnt gegen alle einfachen Strategien, auch gegen `zufall`, und hält gegen das Gleichgewicht mit (3 zu 6 Siege bei 10 Spielen, der Unterschied ist klein).

**Lösungsüberblick**

| Strategie | Idee | Schwäche |
|---|---|---|
| `konstant6` | sicher, immer 6 | wird mit 11 geschlagen |
| `fuenfmehr` | 5 mehr als der Gegner zuletzt | wird mit „6 weniger“ geschlagen |
| `sechsweniger` | 6 weniger als der Gegner zuletzt | wird von `adaptiv` geschlagen |
| `durchschnitt` | Regel bezogen auf den Durchschnitt | träge, schlägt kaum etwas |
| `zufall` | gleichverteilt 1 bis 11 | wird von besseren Mischungen und `adaptiv` geschlagen |
| `gleichgewicht` | gemischte Strategie, nicht auszunutzen | nutzt Schwächen des Gegners kaum |
| `adaptiv` | beste Antwort auf die letzten Einsätze | gegen das Gleichgewicht nur ausgeglichen |

## Zum Schluss: MiniZinc?

Wir haben MiniZinc hier nicht eingesetzt. Die Aufgabe hat keine Eingabedaten, man schreibt eine Strategie, die gegen unbekannte Gegner spielt. Das ist eine Simulation mit Zufall und keine Frage mit festen Eingaben, auf die ein Constraint-Solver eine Antwort geben könnte. Das einzige Rechenproblem darin, das Gleichgewicht der Einzelrunde, sucht Wahrscheinlichkeiten (Kommazahlen); MiniZinc rechnet mit ganzen Zahlen und eignet sich für diese Art Problem (lineares Programm) nicht.

## Für Neugierige

- **Exaktes Gleichgewicht:** Das Gleichgewicht ist die Lösung eines linearen Programms (zum Beispiel mit `scipy.optimize.linprog`): Wahrscheinlichkeiten `p[b]`, die sich zu 1 summieren und für jeden festen Einsatz `e` die Auszahlung `Σ p[b] · auszahlung(e, b) ≤ 0` ergeben. Fictitious play liefert die Näherung ohne Zusatzpaket.
- **Mehr Gedächtnis:** Unsere adaptive Strategie schaut nur auf die Verteilung der letzten Einsätze des Gegners. Die Hinweise empfehlen, auch zu prüfen, ob der Gegner vom vorletzten Einsatz (seinem oder unserem) abhängt, und mehrere Verfahren zu mischen.
- **Zufällig weniger setzen:** Die Hinweise nennen die Variante, eine Basisstrategie mit einem kleinen zufälligen Abschlag zu spielen; so wird sie schwerer zu erkennen.
- **Mehr Gegner:** Ein größeres Turnier (mehr Strategien, mehr Spiele pro Paar) zeigt zuverlässiger, welche Strategie wirklich gut ist; mit 10 Spielen pro Paar schwanken die Ergebnisse noch.
- **Das echte Turnier:** Im Turniersystem hat man nach jeder Runde nur den letzten Einsatz des Gegners gesehen; die Einsatzliste, die wir hier übergeben, hätte man sich dort selbst merken müssen.